In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ---------------------- Step 1: Load Historical LULC Data (2000–2024) ----------------------
historical_data = {
    "Year": [2000, 2005, 2010, 2015, 2020, 2024],
    "Water": [1.72, 1.75, 1.70, 1.65, 1.93, 1.74],
    "Built_up": [20.11, 43.01, 69.48, 89.41, 99.23, 128.54],
    "Bare_Land": [8.03, 8.18, 7.98, 8.70, 6.40, 5.04],
    "Cropland_&_Open_Vegetation": [330.32, 312.21, 288.57, 270.01, 258.10, 230.28],
    "Forest": [56.38, 51.69, 48.77, 46.25, 43.11, 41.98],
    "Wetlands": [13.44, 13.16, 13.50, 13.98, 21.23, 22.42]
}
df_hist = pd.DataFrame(historical_data)

# ---------------------- Step 2: Define Prediction Rules & Rates ----------------------
# Calculate 2000–2024 annual rates (for reference)
annual_rates_2000_2024 = {}
for cls in ["Water", "Built_up", "Bare_Land", "Cropland_&_Open_Vegetation", "Forest", "Wetlands"]:
    total_change = df_hist[cls].iloc[-1] - df_hist[cls].iloc[0]
    annual_rates_2000_2024[cls] = total_change / (2024 - 2000)

# Define 2024–2032 prediction rates (per your rules)
prediction_rates = {
    "Water": 0.0,  # Constant
    "Built_up": annual_rates_2000_2024["Built_up"] * 1.05,  # +5%
    "Forest": annual_rates_2000_2024["Forest"] * 0.5,  # 50% slower loss
    "Wetlands": annual_rates_2000_2024["Wetlands"] * 0.5,  # 50% slower growth
    "Bare_Land": lambda year: annual_rates_2000_2024["Bare_Land"] if year < 2030 else 0.0  # Stabilize after 2030
}

# ---------------------- Step 3: Predict 2028 & 2032 ----------------------
pred_years = [2028, 2032]
df_pred = pd.DataFrame({"Year": pred_years})

# Predict each class (except Cropland_&_Open_Vegetation, which is residual)
for cls in ["Water", "Built_up", "Forest", "Wetlands"]:
    df_pred[cls] = df_hist[cls].iloc[-1] + (df_pred["Year"] - 2024) * prediction_rates[cls]

# Predict Bare Land (stabilizes after 2030)
df_pred["Bare_Land"] = df_hist["Bare_Land"].iloc[-1]
for idx, year in enumerate(df_pred["Year"]):
    years_since_2024 = year - 2024
    if years_since_2024 <= 6:  # 2024→2030: 6 years of decline
        decline_years = min(years_since_2024, 6)
        df_pred.loc[idx, "Bare_Land"] += decline_years * annual_rates_2000_2024["Bare_Land"]
    # 2030→2032: no change

# Predict Cropland_&_Open_Vegetation (residual to keep total = 430 km²)
other_classes = ["Water", "Built_up", "Bare_Land", "Forest", "Wetlands"]
df_pred["Cropland_&_Open_Vegetation"] = 430.0 - df_pred[other_classes].sum(axis=1)

# Round to 2 decimal places (consistent with historical data)
df_pred = df_pred.round(2)

# ---------------------- Step 4: Combine Historical + Predicted Data ----------------------
df_combined = pd.concat([df_hist, df_pred], ignore_index=True).sort_values("Year")

# ---------------------- Step 5: Print Results ----------------------
print("Gasabo LULC Prediction (2028–2032) – All Classes:")
print(df_combined[["Year", "Water", "Built_up", "Bare_Land", "Cropland_&_Open_Vegetation", "Forest", "Wetlands"]].to_string(index=False))

# ---------------------- Step 6: Visualize All Classes ----------------------
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(12, 8))

# Define colors for each class
colors = {
    "Water": "blue",
    "Built_up": "red",
    "Bare_Land": "orange",
    "Cropland_&_Open_Vegetation": "lightgreen",
    "Forest": "darkgreen",
    "Wetlands": "purple"
}

# Plot historical (solid lines) and predicted (dashed lines)
for cls in colors.keys():
    # Historical data (2000–2024)
    ax.plot(
        df_hist["Year"], df_hist[cls], 
        marker="o", color=colors[cls], linewidth=2.5, label=f"{cls} (Observed)"
    )
    # Predicted data (2028–2032)
    ax.plot(
        df_pred["Year"], df_pred[cls], 
        marker="s", color=colors[cls], linewidth=2.5, linestyle="--", label=f"{cls} (Projected)"
    )

# Customize plot
ax.set_xlabel("Year", fontsize=14, fontweight="bold")
ax.set_ylabel("LULC Area (km²)", fontsize=14, fontweight="bold")
ax.set_title("Gasabo LULC Trends: Observed (2000–2024) & Projected (2028–2032) – All Classes", fontsize=16, fontweight="bold")
ax.set_xticks(df_combined["Year"])
ax.tick_params(axis="both", labelsize=12)

# Add legend (2 columns for readability)
ax.legend(ncol=2, fontsize=11, loc="center left", bbox_to_anchor=(1, 0.5))

# Add total area annotation
ax.text(
    0.02, 0.98,
    "Total Area: 430 km²",
    transform=ax.transAxes, ha="left", va="top",
    fontsize=11, fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8)
)

plt.tight_layout()
plt.savefig("Gasabo_LULC_AllClasses_2000_2032.png", dpi=300, bbox_inches="tight")
plt.close()

print("\nVisualization saved as 'Gasabo_LULC_AllClasses_2000_2032.png'")

Gasabo LULC Prediction (2028–2032) – All Classes:
 Year  Water  Built_up  Bare_Land  Cropland_&_Open_Vegetation  Forest  Wetlands
 2000   1.72     20.11       8.03                      330.32   56.38     13.44
 2005   1.75     43.01       8.18                      312.21   51.69     13.16
 2010   1.70     69.48       7.98                      288.57   48.77     13.50
 2015   1.65     89.41       8.70                      270.01   46.25     13.98
 2020   1.93     99.23       6.40                      258.10   43.11     21.23
 2024   1.74    128.54       5.04                      230.28   41.98     22.42
 2028   1.74    147.52       4.54                      212.25   40.78     23.17
 2032   1.74    166.49       5.04                      193.23   39.58     23.92

Visualization saved as 'Gasabo_LULC_AllClasses_2000_2032.png'
